# Station continuity breaks data preparation

In [17]:
from pathlib import Path
import pandas as pd

In [18]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"
DATA_FILE = DATA_DIR / "final_trip_data.parquet"
rides_data = pd.read_parquet(DATA_FILE)

In [19]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
0,2.0,CLASSIC,2024-01-11 07:58:00,51.515428,-0.023565,300048.0,"Stainsby Road , Poplar",5.740013,2024-01-11 07:48:00,Thursday,51.518811,-0.011662,200254,"Teviot Street, Poplar",0.905466,9m 27s,9.464783,136600234
1,2.0,CLASSIC,2024-01-11 08:50:00,51.505224,-0.098032,300208.0,"Lavington Street, Bankside",13.056512,2024-01-11 08:25:00,Thursday,51.515428,-0.023565,300048,"Stainsby Road , Poplar",5.276887,24m 14s,24.249450,136601943
2,2.0,CLASSIC,2024-01-11 15:37:00,51.541603,-0.138853,200105.0,"Bonny Street, Camden Town",9.913698,2024-01-11 15:07:00,Thursday,51.505224,-0.098032,300208,"Lavington Street, Bankside",4.933497,29m 51s,29.858667,136609820
3,2.0,CLASSIC,2024-01-12 10:53:00,51.522482,-0.154907,1194.0,"Westminster University, Marylebone",13.071968,2024-01-12 10:42:00,Friday,51.541603,-0.138853,200105,"Bonny Street, Camden Town",2.398717,11m 0s,11.010050,136625641
4,2.0,CLASSIC,2024-01-13 12:02:00,51.530529,-0.167515,1091.0,"St. John's Wood Church, The Regent's Park",2.930164,2024-01-13 11:36:00,Saturday,51.522482,-0.154907,1194,"Westminster University, Marylebone",1.249569,25m 35s,25.587017,136642769


In [20]:
all_bikes_id = rides_data["bike_id"].unique()

In [21]:
len(all_bikes_id)

15739

In [22]:
# data_preparation.ipynb sorts trips before writing final_trip_data.parquet.
for date_column in ["start_date", "end_date"]:
    rides_data[date_column] = pd.to_datetime(
        rides_data[date_column], format="ISO8601", errors="raise"
    )

if rides_data[["bike_id", "start_date", "end_date"]].isna().any().any():
    raise ValueError("Continuity detection requires bike IDs and valid trip dates.")

def validate_trip_order(trips, batch_size=250_000):
    """Check the saved sort order using small, overlapping batches."""
    for start in range(1, len(trips), batch_size):
        block = trips.iloc[start - 1:start + batch_size]
        bike = block["bike_id"].to_numpy()
        begin = block["start_date"].to_numpy()
        end = block["end_date"].to_numpy()
        decreasing = (
            (bike[1:] < bike[:-1])
            | ((bike[1:] == bike[:-1]) & (
                (begin[1:] < begin[:-1])
                | ((begin[1:] == begin[:-1]) & (end[1:] < end[:-1]))
            ))
        )
        if decreasing.any():
            raise ValueError(
                "Trips are not sorted by bike_id, start_date, end_date. "
                "Run data_preparation.ipynb to regenerate final_trip_data.parquet."
            )


validate_trip_order(rides_data)

# ---------- Previous trip information for the same bike ----------
rides_data["previous_end_station_id"] = (
    rides_data
    .groupby("bike_id")["end_station_id"]
    .shift(1)
)

rides_data["previous_end_date"] = (
    rides_data
    .groupby("bike_id")["end_date"]
    .shift(1)
)
rides_data["previous_end_station_name"] = (
    rides_data
    .groupby("bike_id")["end_station_name"]
    .shift(1)
)

In [23]:
rides_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,...,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id,previous_end_station_id,previous_end_date,previous_end_station_name
0,2.0,CLASSIC,2024-01-11 07:58:00,51.515428,-0.023565,300048.0,"Stainsby Road , Poplar",5.740013,2024-01-11 07:48:00,Thursday,...,-0.011662,200254,"Teviot Street, Poplar",0.905466,9m 27s,9.464783,136600234,NaN,NaT,NaN
1,2.0,CLASSIC,2024-01-11 08:50:00,51.505224,-0.098032,300208.0,"Lavington Street, Bankside",13.056512,2024-01-11 08:25:00,Thursday,...,-0.023565,300048,"Stainsby Road , Poplar",5.276887,24m 14s,24.249450,136601943,300048.0,2024-01-11 07:58:00,"Stainsby Road , Poplar"
2,2.0,CLASSIC,2024-01-11 15:37:00,51.541603,-0.138853,200105.0,"Bonny Street, Camden Town",9.913698,2024-01-11 15:07:00,Thursday,...,-0.098032,300208,"Lavington Street, Bankside",4.933497,29m 51s,29.858667,136609820,300208.0,2024-01-11 08:50:00,"Lavington Street, Bankside"
3,2.0,CLASSIC,2024-01-12 10:53:00,51.522482,-0.154907,1194.0,"Westminster University, Marylebone",13.071968,2024-01-12 10:42:00,Friday,...,-0.138853,200105,"Bonny Street, Camden Town",2.398717,11m 0s,11.010050,136625641,200105.0,2024-01-11 15:37:00,"Bonny Street, Camden Town"
4,2.0,CLASSIC,2024-01-13 12:02:00,51.530529,-0.167515,1091.0,"St. John's Wood Church, The Regent's Park",2.930164,2024-01-13 11:36:00,Saturday,...,-0.154907,1194,"Westminster University, Marylebone",1.249569,25m 35s,25.587017,136642769,1194.0,2024-01-12 10:53:00,"Westminster University, Marylebone"


# ---------- Detecting discontinuities ----------


In [24]:

discontinuity_mask = (
    rides_data["previous_end_station_id"].notna()
    & (rides_data["previous_end_station_id"] != rides_data["start_station_id"])
)

bike_discontinuities = rides_data[discontinuity_mask]

In [25]:
bike_discontinuities["time_gap"] = (
    bike_discontinuities["start_date"] - bike_discontinuities["previous_end_date"]
)

In [26]:
bike_discontinuities["time_gap_minutes"] = (
    bike_discontinuities["time_gap"].dt.total_seconds() / 60
)

In [27]:
bike_discontinuities_table = bike_discontinuities[
    [
        "bike_id",
        "bike_model",
        "previous_end_date",
        "previous_end_station_id",
        "previous_end_station_name",
        "start_date",
        "start_station_id",
        "start_station_name",
        "end_date",
        "end_station_id",
        "end_station_name",
        "time_gap",
        "time_gap_minutes",
        "trip_id"
    ]
]

In [28]:
bike_discontinuities_table.head(20)

,bike_id,bike_model,previous_end_date,previous_end_station_id,previous_end_station_name,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,time_gap,time_gap_minutes,trip_id
7,2.0,CLASSIC,2024-01-15 15:24:00,200005.0,"New Cavendish Street, Marylebone",2024-01-15 17:45:00,300235,"York Way, Kings Cross",2024-01-15 17:56:00,972.0,"Hampstead Road, Euston",0 days 02:21:00,141.0,136681171
15,2.0,CLASSIC,2024-01-17 08:40:00,200072.0,"Houndsditch, Aldgate",2024-01-18 18:15:00,200021,"Hermitage Court, Wapping",2024-01-18 18:17:00,200021.0,"Hermitage Court, Wapping",1 days 09:35:00,2015.0,136749073
34,2.0,CLASSIC,2024-01-29 19:29:00,10624.0,"Strata, Elephant & Castle",2024-01-31 14:39:00,1086,"Baker Street, Marylebone",2024-01-31 15:14:00,1086.0,"Baker Street, Marylebone",1 days 19:10:00,2590.0,137007661
47,2.0,CLASSIC,2024-02-04 10:04:00,200244.0,"Hollybush Gardens, Bethnal Green",2024-02-09 17:28:00,1147,"Clifton Road, Maida Vale",2024-02-09 17:35:00,200110.0,"Bishop's Bridge Road West, Bayswater",5 days 07:24:00,7644.0,137199288
50,2.0,CLASSIC,2024-02-11 14:11:00,200128.0,"Queen Street 2, Bank",2024-02-17 12:26:00,1057,"Hardwick Street, Clerkenwell",2024-02-17 12:36:00,3456.0,"Portugal Street, Holborn",5 days 22:15:00,8535.0,137356872
94,2.0,CLASSIC,2024-03-04 17:24:00,962.0,"Stamford Street, South Bank",2024-03-05 17:27:00,300092,"Hertford Road, De Beauvoir Town",2024-03-05 17:43:00,300092.0,"Hertford Road, De Beauvoir Town",1 days 00:03:00,1443.0,137687246
100,2.0,CLASSIC,2024-03-08 07:27:00,999.0,"Queen Street 1, Bank",2024-03-08 16:36:00,1159,"Berry Street, Clerkenwell",2024-03-08 16:56:00,200247.0,"Globe Town Market, Bethnal Green",0 days 09:09:00,549.0,137762479
103,2.0,CLASSIC,2024-03-09 13:33:00,200014.0,"Gunmakers Lane, Old Ford",2024-03-11 07:30:00,200252,"Bow Church Station, Bow",2024-03-11 07:55:00,200203.0,"King Edward Street, St Pauls",1 days 17:57:00,2517.0,137801377
128,2.0,CLASSIC,2024-03-29 23:09:00,200202.0,"Storey's Gate, Westminster",2024-03-30 14:13:00,2677,"Natural History Museum, South Kensington",2024-03-30 14:52:00,1199.0,"The Tennis Courts, The Regent's Park",0 days 15:04:00,904.0,138235017
144,2.0,CLASSIC,2024-04-10 20:38:00,200206.0,"Chicheley Street, South Bank",2024-04-11 14:23:00,200144,"Sea Containers, South Bank",2024-04-11 14:34:00,1053.0,"Clerkenwell Green, Clerkenwell",0 days 17:45:00,1065.0,138489608


In [29]:
print(f"Number of discontinuities: {len(bike_discontinuities_table)}")

Number of discontinuities: 1044959


In [30]:
discontinuity_percent = (
    len(bike_discontinuities_table) / len(rides_data) * 100
)

print(f"Percentage of discontinuous trips: {discontinuity_percent:.3f}%")

Percentage of discontinuous trips: 5.996%


In [31]:
rides_data_continuous = rides_data[~discontinuity_mask]

In [32]:
rides_data_continuous.to_parquet(DATA_DIR /
    "rides_data_without_discontinuities.parquet",
    index=False
)